# ECG Project — 02. Dataset audit and label analysis

Продолжаем работу после скачивания данных. Этот ноутбук **не скачивает и не удаляет датасеты**: он проверяет Google Drive, инвентаризирует файлы и начинает аудит меток.

План: Chapman–Shaoxing → аудит меток → PTB-XL metadata → проверка patient-level folds → отчёт. Не объединяем датасеты, пока не согласуем семантику меток и уровень аннотации.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path
import os, json
from collections import Counter, defaultdict
from datetime import datetime

PROJECT_ROOT = Path('/content/drive/MyDrive/ECG_detection')
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
REPORTS_ROOT = PROJECT_ROOT / 'reports'
print('Project exists:', PROJECT_ROOT.exists(), PROJECT_ROOT)
print('Raw data exists:', RAW_ROOT.exists(), RAW_ROOT)

Project exists: True /content/drive/MyDrive/ECG_detection
Raw data exists: True /content/drive/MyDrive/ECG_detection/data/raw


## 1. Инвентаризация структуры
Выводим каталоги и количество файлов, не загружая сигналы в память.

In [5]:
def show_tree(root: Path, max_depth=4):
    if not root.exists():
        print(f'[NOT FOUND] {root}')
        return
    root_depth = len(root.parts)
    for current, dirs, files in os.walk(root):
        p = Path(current)
        depth = len(p.parts) - root_depth
        if depth >= max_depth:
            dirs[:] = []
        print('  ' * depth + f'{p.name}/ [{len(files)} files]')

show_tree(PROJECT_ROOT, max_depth=4)

ECG_detection/ [0 files]
  data/ [0 files]
    raw/ [0 files]
      chapman/ [0 files]
        physionet.org/ [1 files]
      ptbxl/ [0 files]
        physionet.org/ [1 files]
      incart/ [0 files]
        physionet.org/ [1 files]
      mitbih/ [0 files]
        physionet.org/ [1 files]
    processed/ [0 files]
    external/ [0 files]
  notebooks/ [0 files]
  src/ [0 files]
  reports/ [0 files]


In [6]:
def inventory(folder: Path):
    counts = Counter()
    examples = defaultdict(list)
    if folder.exists():
        for p in folder.rglob('*'):
            if p.is_file():
                ext = p.suffix.lower() or '[no extension]'
                counts[ext] += 1
                if len(examples[ext]) < 3:
                    examples[ext].append(str(p.relative_to(folder)))
    return counts, examples

if RAW_ROOT.exists():
    for folder in sorted(p for p in RAW_ROOT.iterdir() if p.is_dir()):
        counts, examples = inventory(folder)
        print(f'\\n### {folder.name}: {sum(counts.values())} files')
        for ext, n in counts.most_common():
            print(f'{ext:16} {n:8}  {examples[ext] if ext in {".hea", ".csv", ".mat", ".dat"} else ""}')
else:
    print('Проверь путь PROJECT_ROOT и фактическое имя папки в Google Drive.')

\n### chapman: 91261 files
.hea                45152  ['physionet.org/files/ecg-arrhythmia/1.0.0/WFDBRecords/46/460/JS45485.hea', 'physionet.org/files/ecg-arrhythmia/1.0.0/WFDBRecords/46/460/JS45466.hea', 'physionet.org/files/ecg-arrhythmia/1.0.0/WFDBRecords/46/460/JS45425.hea']
.mat                45152  ['physionet.org/files/ecg-arrhythmia/1.0.0/WFDBRecords/46/460/JS45484.mat', 'physionet.org/files/ecg-arrhythmia/1.0.0/WFDBRecords/46/460/JS45499.mat', 'physionet.org/files/ecg-arrhythmia/1.0.0/WFDBRecords/46/460/JS45453.mat']
.html                 500  
[no extension]        453  
.txt                    3  
.csv                    1  ['physionet.org/files/ecg-arrhythmia/1.0.0/ConditionNames_SNOMED-CT.csv']
\n### incart: 233 files
.dat                   75  ['physionet.org/files/incartdb/1.0.0/I63.dat', 'physionet.org/files/incartdb/1.0.0/I64.dat', 'physionet.org/files/incartdb/1.0.0/I05.dat']
.atr                   75  
.hea                   75  ['physionet.org/files/incartdb/1.0.0/

## 2. Установка библиотек для метаданных и WFDB

In [7]:
%pip -q install wfdb pandas numpy scipy scikit-learn matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 5.9 MB/s eta 0:00:00


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
try:
    import wfdb
    print('wfdb version:', wfdb.__version__)
except Exception as e:
    print('WFDB import error:', repr(e))

wfdb version: 4.3.1


## 3. Chapman–Shaoxing: поиск записей и чтение заголовков

WFDB-запись обычно состоит из `.hea` и `.dat`. Читаем только несколько заголовков для проверки числа отведений, частоты дискретизации и длины записи.

In [9]:
chapman_candidates = []
if RAW_ROOT.exists():
    for d in RAW_ROOT.iterdir():
        if d.is_dir() and any(k in d.name.lower() for k in ['chapman', 'shaoxing', 'ecg-arrhythmia']):
            chapman_candidates.append(d)

print('Candidate folders:', [str(p) for p in chapman_candidates])
chapman_root = chapman_candidates[0] if chapman_candidates else None

if chapman_root:
    hea_files = list(chapman_root.rglob('*.hea'))
    print('Header files:', len(hea_files))
    rows = []
    for hp in hea_files[:10]:
        try:
            h = wfdb.rdheader(str(hp.with_suffix('')))
            rows.append({'header': str(hp.relative_to(chapman_root)), 'leads': h.n_sig,
                         'sampling_rate_hz': h.fs, 'samples': h.sig_len,
                         'duration_seconds': h.sig_len / h.fs if h.fs else None,
                         'lead_names': ', '.join(h.sig_name or [])})
        except Exception as e:
            rows.append({'header': str(hp.relative_to(chapman_root)), 'error': str(e)})
    display(pd.DataFrame(rows))
else:
    print('Каталог Chapman не найден автоматически. Исправь путь вручную по дереву выше.')

Candidate folders: ['/content/drive/MyDrive/ECG_detection/data/raw/chapman']
Header files: 45152


,header,leads,sampling_rate_hz,samples,duration_seconds,lead_names
0,physionet.org/files/ecg-arrhythmia/1.0.0/WFDBR...,12,500,5000,10.0,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6"
1,physionet.org/files/ecg-arrhythmia/1.0.0/WFDBR...,12,500,5000,10.0,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6"
2,physionet.org/files/ecg-arrhythmia/1.0.0/WFDBR...,12,500,5000,10.0,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6"
3,physionet.org/files/ecg-arrhythmia/1.0.0/WFDBR...,12,500,5000,10.0,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6"
4,physionet.org/files/ecg-arrhythmia/1.0.0/WFDBR...,12,500,5000,10.0,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6"
5,physionet.org/files/ecg-arrhythmia/1.0.0/WFDBR...,12,500,5000,10.0,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6"
6,physionet.org/files/ecg-arrhythmia/1.0.0/WFDBR...,12,500,5000,10.0,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6"
7,physionet.org/files/ecg-arrhythmia/1.0.0/WFDBR...,12,500,5000,10.0,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6"
8,physionet.org/files/ecg-arrhythmia/1.0.0/WFDBR...,12,500,5000,10.0,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6"
9,physionet.org/files/ecg-arrhythmia/1.0.0/WFDBR...,12,500,5000,10.0,"I, II, III, aVR, aVL, aVF, V1, V2, V3, V4, V5, V6"


## 4. PTB-XL: metadata и диагностические коды

Ищем `ptbxl_database.csv` и `scp_statements.csv` внутри папок, имя которых содержит `ptb`. Никаких загрузок из сети.

In [10]:
def find_named_file(root: Path, filename: str):
    matches = list(root.rglob(filename)) if root.exists() else []
    return matches[0] if matches else None

ptb_roots = [d for d in RAW_ROOT.iterdir() if d.is_dir() and 'ptb' in d.name.lower()] if RAW_ROOT.exists() else []
print('PTB folders:', [str(p) for p in ptb_roots])

ptb_db_path = next((find_named_file(r, 'ptbxl_database.csv') for r in ptb_roots
                    if find_named_file(r, 'ptbxl_database.csv')), None)
scp_path = next((find_named_file(r, 'scp_statements.csv') for r in ptb_roots
                 if find_named_file(r, 'scp_statements.csv')), None)
print('Database metadata:', ptb_db_path)
print('SCP statements:', scp_path)

ptbxl = pd.read_csv(ptb_db_path) if ptb_db_path else None
scp_statements = pd.read_csv(scp_path, index_col=0) if scp_path else None
if ptbxl is not None:
    print('PTB-XL shape:', ptbxl.shape)
    display(ptbxl.head())
    print('Columns:', ptbxl.columns.tolist())
else:
    print('PTB-XL metadata не найдена — проверь структуру и распаковку.')
if scp_statements is not None:
    print('SCP statements shape:', scp_statements.shape)
    display(scp_statements.head())

PTB folders: ['/content/drive/MyDrive/ECG_detection/data/raw/ptbxl']
Database metadata: /content/drive/MyDrive/ECG_detection/data/raw/ptbxl/physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv
SCP statements: /content/drive/MyDrive/ECG_detection/data/raw/ptbxl/physionet.org/files/ptb-xl/1.0.3/scp_statements.csv
PTB-XL shape: (21799, 28)


,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr


Columns: ['ecg_id', 'patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']
SCP statements shape: (71, 12)


,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,non-diagnostic T abnormalities,NaN,NaN,NaN,NaN
NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_,Basic roots for coding ST-T changes and abnorm...,non-specific ST changes,145.0,MDC_ECG_RHY_STHILOST,NaN,NaN
DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,suggests digitalis-effect,205.0,NaN,NaN,NaN
LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,long QT-interval,148.0,NaN,NaN,NaN
NORM,normal ECG,1.0,NaN,NaN,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7


## 5. PTB-XL: распределение кодов и patient-level folds

In [11]:
if ptbxl is not None:
    if 'patient_id' in ptbxl.columns:
        print('Unique patients:', ptbxl['patient_id'].nunique())
    if 'strat_fold' in ptbxl.columns:
        print('Records by recommended strat_fold:')
        display(ptbxl['strat_fold'].value_counts().sort_index().rename('records').to_frame())
    if 'scp_codes' in ptbxl.columns:
        def parse_codes(x):
            if pd.isna(x):
                return {}
            if isinstance(x, dict):
                return x
            try:
                return ast.literal_eval(x)
            except Exception:
                return {}
        code_counts = Counter()
        for x in ptbxl['scp_codes']:
            code_counts.update(parse_codes(x).keys())
        display(pd.DataFrame(code_counts.most_common(50), columns=['SCP_code', 'records_with_code']))
    else:
        print('Колонка scp_codes отсутствует.')

Unique patients: 18869
Records by recommended strat_fold:


,records
strat_fold,
1,2175
2,2181
3,2192
4,2174
5,2174
6,2173
7,2176
8,2173
9,2183


,SCP_code,records_with_code
0,SR,16748
1,NORM,9514
2,ABQRS,3327
3,IMI,2676
4,ASMI,2357
5,LVH,2132
6,NDT,1825
7,LAFB,1623
8,AFIB,1514
9,ISC_,1272


## 6. Сохранить отчёт инвентаризации

Отчёт хранит только структуру и количество файлов, не сами ЭКГ.

In [12]:
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)
audit = {
    'created_at_utc': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    'project_root': str(PROJECT_ROOT),
    'project_root_exists': PROJECT_ROOT.exists(),
    'raw_root_exists': RAW_ROOT.exists(),
    'datasets': []
}
if RAW_ROOT.exists():
    for folder in sorted(p for p in RAW_ROOT.iterdir() if p.is_dir()):
        counts, _ = inventory(folder)
        audit['datasets'].append({'name': folder.name, 'path': str(folder),
                                  'file_count': int(sum(counts.values())),
                                  'extensions': dict(counts)})
audit_path = REPORTS_ROOT / 'dataset_inventory.json'
with open(audit_path, 'w', encoding='utf-8') as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)
print('Saved:', audit_path)

/tmp/ipykernel_2108/2306660674.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at_utc': datetime.utcnow().isoformat(timespec='seconds') + 'Z',


Saved: /content/drive/MyDrive/ECG_detection/reports/dataset_inventory.json


## 7. Следующий шаг

После запуска проверь:
1. Что `PROJECT_ROOT` найден и все каталоги датасетов отображаются.
2. Что Chapman содержит ожидаемые `.hea`/`.dat` или иной фактический формат.
3. Что PTB-XL metadata прочиталась и видны `patient_id`, `strat_fold`, `scp_codes`.

Далее создадим отдельный notebook для воспроизведения baseline Chapman и анализа распределения исходных классов. Не делаем случайный split по записям: он может допустить утечку между пациентами.